In [ ]:
# Celda 1: setup del entorno del notebook
from functions.notebook_setup import setup_environment

# Configura semilla, rutas del proyecto y path para imports
project_root, data_dir, functions_dir = setup_environment(
    seed=42,
    marker_dir="functions",
    verbose=True,
)

print(f"project_root: {project_root}")
print(f"data_dir: {data_dir}")
print(f"functions_dir: {functions_dir}")


In [ ]:
# Celda 2: imports base
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# Celda 3: carga de datos de pozo y estadística básica
log_file = Path(data_dir) / "B-41_logs.txt"

# Lee datos numéricos a partir de la sección ~Ascii Data Section
rows = []
in_ascii = False
with open(log_file, "r", encoding="utf-8") as f:
    for line in f:
        txt = line.strip()
        if not in_ascii:
            if txt.upper().startswith("~ASCII DATA SECTION"):
                in_ascii = True
            continue
        if not txt:
            continue
        parts = txt.split()
        if len(parts) < 4:
            continue
        # Columnas esperadas: DEPT, DT, GR, RHOB, (opcional PVEL)
        rows.append(parts[:5])

well_df = pd.DataFrame(rows, columns=["DEPT", "DT", "GR", "RHOB", "PVEL"])
well_df = well_df.apply(pd.to_numeric, errors="coerce")

# Reemplaza muestras vacías
well_df = well_df.replace(-999.25, np.nan)

# Conserva logs solicitados + profundidad
well_logs_df = well_df[["DEPT", "GR", "RHOB", "DT"]].copy()

# Estadística básica de logs y profundidad
stats_df = well_logs_df.agg(["min", "max", "mean"]).T
stats_df = stats_df.rename(columns={"mean": "average"})

# Estadística de muestreo vertical (diferencia de profundidad punto a punto)
dz = well_logs_df["DEPT"].diff().dropna()
dz_stats_df = dz.agg(["min", "max", "mean"]).to_frame(name="delta_depth_m").T
dz_stats_df = dz_stats_df.rename(columns={"mean": "average"})

print(f"Archivo cargado: {log_file}")
print(f"Filas: {len(well_logs_df):,}")
print("\nValores nulos por columna:")
print(well_logs_df.isna().sum())
print("\nEstadística básica (min, max, average):")
display(stats_df)
print("\nEstadística del muestreo en profundidad (Δz, min/max/average):")
display(dz_stats_df)

well_logs_df.head()

In [ ]:
# Celda 4: control visual de logs (3 tracks)
import matplotlib.pyplot as plt

# Datos ordenados por profundidad
df_plot = well_logs_df.sort_values("DEPT").copy()

depth = df_plot["DEPT"].to_numpy()
gr = df_plot["GR"].to_numpy()
rhob = df_plot["RHOB"].to_numpy()
dt = df_plot["DT"].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 10), sharey=True)

# Track 1: Gamma Ray (verde, 0-200)
axes[0].plot(gr, depth, color="green", lw=1.0)
axes[0].set_xlim(0, 200)
axes[0].set_xlabel("Gamma Ray")
axes[0].set_title("GR")

# Track 2: Densidad (rojo, 1.95-2.95)
axes[1].plot(rhob, depth, color="red", lw=1.0)
axes[1].set_xlim(1.95, 2.95)
axes[1].set_xlabel("Densidad (g/cc)")
axes[1].set_title("RHOB")

# Track 3: Sónico (azul, 140-15 invertido en eje x)
axes[2].plot(dt, depth, color="blue", lw=1.0)
axes[2].set_xlim(140, 15)
axes[2].set_xlabel("Sónico DT (us/ft)")
axes[2].set_title("DT")

# Profundidad común fija y guías cada 500 m
depth_min = 0.0
depth_max = 3800.0
for ax in axes:
    ax.set_ylim(depth_min, depth_max)
    ax.invert_yaxis()
    for y in np.arange(depth_min, depth_max + 500.0, 500.0):
        ax.axhline(y, color="gray", lw=0.6, alpha=0.6)
    ax.grid(False)

axes[0].set_ylabel("Profundidad (m)")
plt.suptitle("Control de calidad de logs de pozo", y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 5: cálculo de velocidad VEL (m/s) desde sónico DT (us/ft)
# Conversión: VEL[m/s] = 304800 / DT[us/ft]
well_logs_df["VEL"] = 304800.0 / well_logs_df["DT"]

# Maneja valores no físicos (DT<=0) y no finitos
well_logs_df.loc[well_logs_df["DT"] <= 0, "VEL"] = np.nan
well_logs_df.loc[~np.isfinite(well_logs_df["VEL"]), "VEL"] = np.nan

print("Resumen VEL (m/s):")
display(well_logs_df[["VEL"]].agg(["min", "max", "mean"]).rename(index={"mean": "average"}))

well_logs_df[["DEPT", "DT", "VEL"]].head()

In [ ]:
# Celda 6: plot de velocidad (track único)
import matplotlib.pyplot as plt

# Datos ordenados por profundidad
df_vel_plot = well_logs_df.sort_values("DEPT").copy()

depth = df_vel_plot["DEPT"].to_numpy()
vel = df_vel_plot["VEL"].to_numpy()

fig, ax = plt.subplots(1, 1, figsize=(4.5, 10))
ax.plot(vel, depth, color="#0b1f5b", lw=1.0)  # azul oscuro
ax.set_xlim(1500, 6000)
ax.set_xlabel("Velocidad VEL (m/s)")
ax.set_title("VEL")

# Misma escala vertical que la celda previa
ax.set_ylim(0.0, 3800.0)
ax.invert_yaxis()
for y in np.arange(0.0, 3800.0 + 500.0, 500.0):
    ax.axhline(y, color="gray", lw=0.6, alpha=0.6)
ax.grid(False)
ax.set_ylabel("Profundidad (m)")

plt.tight_layout()
plt.show()

In [ ]:
# Celda 7: extensión somera de RHOB y VEL hasta z=0 con dz=0.1524
# Nota: RHOB y VEL pueden tener distinta primera muestra válida
rv_df = well_logs_df[["DEPT", "RHOB", "VEL"]].sort_values("DEPT").reset_index(drop=True).copy()

dz_target = 0.1524
z_top_target = 0.0
vel_shallow_fill = 2000.0  # solicitado para zona somera

# Primera profundidad con RHOB válida
rhob_valid = rv_df.dropna(subset=["RHOB"])
if len(rhob_valid) == 0:
    raise ValueError("No hay muestras válidas de RHOB en el dataframe.")
z_top_rhob = float(rhob_valid["DEPT"].iloc[0])
rhob_top = float(rhob_valid["RHOB"].iloc[0])

# Primera profundidad con VEL válida
vel_valid = rv_df.dropna(subset=["VEL"])
if len(vel_valid) == 0:
    raise ValueError("No hay muestras válidas de VEL en el dataframe.")
z_top_vel = float(vel_valid["DEPT"].iloc[0])

# Construye mallas someras independientes para cada log
z_shallow_rhob = np.arange(z_top_target, z_top_rhob, dz_target, dtype=float)
z_shallow_vel = np.arange(z_top_target, z_top_vel, dz_target, dtype=float)

rhob_shallow_df = pd.DataFrame({"DEPT": z_shallow_rhob, "RHOB": rhob_top})
vel_shallow_df = pd.DataFrame({"DEPT": z_shallow_vel, "VEL": vel_shallow_fill})

# Une original + extensiones y consolida por profundidad
rv_extended_df = rv_df.merge(rhob_shallow_df, on="DEPT", how="outer", suffixes=("", "_sh"))
rv_extended_df = rv_extended_df.merge(vel_shallow_df, on="DEPT", how="outer", suffixes=("", "_sh"))

rv_extended_df["RHOB"] = rv_extended_df["RHOB"].fillna(rv_extended_df["RHOB_sh"])
rv_extended_df["VEL"] = rv_extended_df["VEL"].fillna(rv_extended_df["VEL_sh"])
rv_extended_df = rv_extended_df[["DEPT", "RHOB", "VEL"]].sort_values("DEPT").reset_index(drop=True)

# Estadística básica de logs extendidos
stats_rv_ext_df = rv_extended_df[["RHOB", "VEL"]].agg(["min", "max", "mean"]).T
stats_rv_ext_df = stats_rv_ext_df.rename(columns={"mean": "average"})

# Relleno de gaps: interpolación vertical si quedan NaN en RHOB/VEL
for col in ["RHOB", "VEL"]:
    rv_extended_df[col] = rv_extended_df[col].interpolate(method="linear", limit_direction="both")

# QC de nulos en logs extendidos (después del relleno)
null_counts = rv_extended_df[["RHOB", "VEL"]].isna().sum()
null_pct = (null_counts / len(rv_extended_df) * 100.0).round(3)
qc_nulls_df = pd.DataFrame({"n_null": null_counts, "pct_null": null_pct})

rhob_nan_depths = rv_extended_df.loc[rv_extended_df["RHOB"].isna(), "DEPT"]
vel_nan_depths = rv_extended_df.loc[rv_extended_df["VEL"].isna(), "DEPT"]

# Recalcula estadística tras interpolación
stats_rv_ext_df = rv_extended_df[["RHOB", "VEL"]].agg(["min", "max", "mean"]).T
stats_rv_ext_df = stats_rv_ext_df.rename(columns={"mean": "average"})

print(f"z más somera RHOB válida: {z_top_rhob:.4f} m")
print(f"z más somera VEL válida: {z_top_vel:.4f} m")
print(f"Muestras someras añadidas RHOB: {len(z_shallow_rhob)}")
print(f"Muestras someras añadidas VEL: {len(z_shallow_vel)}")
print(f"Valor somero usado para RHOB: {rhob_top:.4f}")
print(f"Valor somero usado para VEL: {vel_shallow_fill:.1f} m/s")

print("\nEstadística básica logs extendidos (min, max, average):")
display(stats_rv_ext_df)

print("\nQC de nulos (RHOB y VEL) tras interpolación:")
display(qc_nulls_df)

print("Primeras profundidades con RHOB NaN (si existen):")
display(rhob_nan_depths.head(20).to_frame(name="DEPT"))
print("Primeras profundidades con VEL NaN (si existen):")
display(vel_nan_depths.head(20).to_frame(name="DEPT"))

print("Primeros 5 valores extendidos:")
display(rv_extended_df.head())

In [ ]:
# Celda 8: plot de RHOB y VEL extendidos (2 tracks)
import matplotlib.pyplot as plt

df_plot_rv = rv_extended_df.sort_values("DEPT").copy()

depth = df_plot_rv["DEPT"].to_numpy()
rhob = df_plot_rv["RHOB"].to_numpy()
vel = df_plot_rv["VEL"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 10), sharey=True)

# Track RHOB
axes[0].plot(rhob, depth, color="red", lw=1.0)
axes[0].set_xlim(1.95, 2.95)
axes[0].set_xlabel("Densidad (g/cc)")
axes[0].set_title("RHOB")

# Track VEL
axes[1].plot(vel, depth, color="#0b1f5b", lw=1.0)  # azul oscuro
axes[1].set_xlim(1500, 6000)
axes[1].set_xlabel("Velocidad (m/s)")
axes[1].set_title("VEL")

# Escala vertical completa: 0 a 3800 m
for ax in axes:
    ax.set_ylim(0.0, 3800.0)
    ax.invert_yaxis()
    for y in np.arange(0.0, 3800.0 + 500.0, 500.0):
        ax.axhline(y, color="gray", lw=0.6, alpha=0.6)
    ax.grid(False)

axes[0].set_ylabel("Profundidad (m)")
plt.suptitle("RHOB y VEL extendidos hasta superficie", y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 9: nuevos atributos VEL_AVG, tiempo vertical (OWT) y AI
# VEL_AVG: velocidad media desde z=0 hasta cada profundidad
# OWT_S/OWT_MS: one-way vertical time de 0 a z
# AI: impedancia acústica = RHOB * VEL
# AI_FILT: AI filtrada con mediana (ventana 20)

work_df = rv_extended_df.sort_values("DEPT").reset_index(drop=True).copy()

# Espesor incremental por muestra (m) en malla dz=0.1524
work_df["DZ"] = work_df["DEPT"].diff().fillna(work_df["DEPT"].iloc[0])
work_df.loc[work_df["DZ"] < 0, "DZ"] = np.nan

# One-way time incremental dt = dz / v y acumulado desde superficie
work_df["DT_S"] = work_df["DZ"] / work_df["VEL"]
work_df["OWT_S"] = work_df["DT_S"].cumsum()
work_df["OWT_MS"] = 1000.0 * work_df["OWT_S"]

# VEL_AVG(z) = z / OWT(z)
work_df["VEL_AVG"] = np.where(work_df["OWT_S"] > 0, work_df["DEPT"] / work_df["OWT_S"], work_df["VEL"])

# Impedancia acústica instantánea
work_df["AI"] = work_df["RHOB"] * work_df["VEL"]

# Filtro de mediana sobre AI (ventana 20 muestras)
work_df["AI_FILT"] = work_df["AI"].rolling(window=20, center=True, min_periods=1).median()

# Dataframe final de trabajo
rv_model_df = work_df[["DEPT", "RHOB", "VEL", "VEL_AVG", "OWT_S", "OWT_MS", "AI", "AI_FILT"]].copy()

print("Estadística básica de nuevos atributos:")
display(rv_model_df[["VEL", "VEL_AVG", "OWT_S", "AI", "AI_FILT"]].agg(["min", "max", "mean"]).rename(index={"mean": "average"}))

rv_model_df.head()

In [ ]:
# Celda 10: plot VEL instantánea vs VEL_AVG + AI (2 tracks)
import matplotlib.pyplot as plt

df_plot = rv_model_df.sort_values("DEPT").copy()

depth = df_plot["DEPT"].to_numpy()
vel_inst = df_plot["VEL"].to_numpy()
vel_avg = df_plot["VEL_AVG"].to_numpy()
ai = df_plot["AI"].to_numpy()
ai_filt = df_plot["AI_FILT"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(9, 10), sharey=True)

# Track 1: velocidades
axes[0].plot(vel_inst, depth, color="#0b1f5b", lw=1.0, label="VEL instantánea")  # azul oscuro
axes[0].plot(vel_avg, depth, color="red", lw=1.0, label="VEL average")
axes[0].set_xlim(1500, 6000)
axes[0].set_xlabel("Velocidad (m/s)")
axes[0].set_title("VEL vs VEL_AVG")
axes[0].legend(loc="best")

# Track 2: impedancia acústica (cruda + filtrada)
axes[1].plot(ai, depth, color="lightgray", lw=1.0, label="AI")
axes[1].plot(ai_filt, depth, color="black", lw=1.0, label="AI filtrada")
axes[1].set_xlabel("AI (kg/m2s aprox)")
axes[1].set_title("Impedancia Acústica")
axes[1].legend(loc="best")

# Misma configuración vertical que antes
for ax in axes:
    ax.set_ylim(0.0, 3800.0)
    ax.invert_yaxis()
    for y in np.arange(0.0, 3800.0 + 500.0, 500.0):
        ax.axhline(y, color="gray", lw=0.6, alpha=0.6)
    ax.grid(False)

axes[0].set_ylabel("Profundidad (m)")
plt.suptitle("Velocidades e Impedancia Acústica", y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 11: definición de geometría VSP + wavelet de fuente
# Pozo vertical, fuente con offset horizontal de 60 m respecto a la cabeza de pozo

source_offset_m = 60.0
receiver_spacing_m = 25.0
first_receiver_depth_m = 25.0
wellhead_x_m = 0.0
wellhead_z_m = 0.0

# Profundidad máxima de receptores: fondo de los registros del pozo
z_max_logs_m = float(rv_model_df["DEPT"].max())
receiver_depths_m = np.arange(first_receiver_depth_m, z_max_logs_m + receiver_spacing_m, receiver_spacing_m, dtype=float)

# Coordenadas fuente y receptores (sistema x-z, z positiva hacia abajo)
source_x_m = source_offset_m
source_z_m = 0.0
receiver_x_m = np.zeros_like(receiver_depths_m)
receiver_z_m = receiver_depths_m.copy()

# Wavelet de fuente: Ricker 20 Hz, duración 128 ms, dt 1 ms
f0_hz = 20.0
dt_wavelet_s = 0.001
wavelet_length_s = 0.128
t_half = wavelet_length_s / 2.0
tw_s = np.arange(-t_half, t_half + dt_wavelet_s, dt_wavelet_s, dtype=float)
p2 = (np.pi**2) * (f0_hz**2) * (tw_s**2)
source_wavelet = (1.0 - 2.0 * p2) * np.exp(-p2)
source_wavelet /= np.max(np.abs(source_wavelet))

vsp_geometry = {
    "source_offset_m": source_offset_m,
    "receiver_spacing_m": receiver_spacing_m,
    "first_receiver_depth_m": first_receiver_depth_m,
    "z_max_logs_m": z_max_logs_m,
    "source_x_m": source_x_m,
    "source_z_m": source_z_m,
    "wellhead_x_m": wellhead_x_m,
    "wellhead_z_m": wellhead_z_m,
    "receiver_x_m": receiver_x_m,
    "receiver_z_m": receiver_z_m,
    "wavelet_f0_hz": f0_hz,
    "wavelet_dt_s": dt_wavelet_s,
    "wavelet_length_s": wavelet_length_s,
    "wavelet_time_s": tw_s,
    "source_wavelet": source_wavelet,
}

print("Geometría VSP definida:")
print(f"- Offset fuente-cabeza de pozo: {source_offset_m:.1f} m")
print(f"- Primer receptor: {first_receiver_depth_m:.1f} m")
print(f"- Espaciado receptores: {receiver_spacing_m:.1f} m")
print(f"- Profundidad máxima logs: {z_max_logs_m:.2f} m")
print(f"- Número de receptores: {len(receiver_depths_m)}")
print("\nWavelet fuente:")
print(f"- Tipo: Ricker")
print(f"- Frecuencia central: {f0_hz:.1f} Hz")
print(f"- Duración: {wavelet_length_s*1000:.1f} ms")
print(f"- Muestreo: {dt_wavelet_s*1000:.1f} ms")
print(f"- Nº muestras: {len(source_wavelet)}")

In [ ]:
# Celda 12: mallado receptor-tiempo para el wavefield VSP
# Filas: receptores, Columnas: muestras de tiempo

# Eje temporal solicitado
dt_seis_s = 0.001          # 1 ms
tmin_s = 0.0
tmax_s = 4.300             # 4300 ms
time_axis_s = np.arange(tmin_s, tmax_s + dt_seis_s, dt_seis_s, dtype=float)
time_axis_ms = time_axis_s * 1000.0

# Eje de receptores (una fila por receptor)
receiver_depths_m = vsp_geometry["receiver_z_m"].astype(float)
n_receivers = len(receiver_depths_m)
nt = len(time_axis_s)

# Mallado base del wavefield (inicializado a 0)
# shape = [n_receptores, n_tiempos]
wavefield_mesh = np.zeros((n_receivers, nt), dtype=float)

# Mallas auxiliares para operaciones vectorizadas
receiver_depth_grid_m, time_grid_s = np.meshgrid(receiver_depths_m, time_axis_s, indexing="ij")

print("Mallado VSP creado:")
print(f"- Receptores (filas): {n_receivers}")
print(f"- Muestras de tiempo (columnas): {nt}")
print(f"- dt: {dt_seis_s*1000:.1f} ms")
print(f"- Ventana temporal: {time_axis_ms[0]:.1f} a {time_axis_ms[-1]:.1f} ms")
print(f"- Shape wavefield_mesh: {wavefield_mesh.shape}")

In [ ]:
# Celda 13: croquis de la geometría VSP
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 10))

# Pozo vertical
ax.plot([wellhead_x_m, wellhead_x_m], [0.0, 3800.0], color="black", lw=1.2, label="Pozo vertical")

# Receptores VSP
ax.scatter(receiver_x_m, receiver_z_m, s=16, color="tab:orange", label="Receptores VSP")

# Cabeza de pozo
ax.scatter([wellhead_x_m], [wellhead_z_m], s=60, marker="s", color="black", label="Cabeza de pozo")

# Fuente
ax.scatter([source_x_m], [source_z_m], s=70, marker="*", color="red", label="Fuente")

# Línea y etiqueta de offset fuente-pozo
ax.plot([wellhead_x_m, source_x_m], [0.0, 0.0], color="gray", lw=1.0, ls="--")
ax.text(source_x_m / 2.0, -70.0, f"Offset = {source_offset_m:.0f} m", ha="center", va="bottom", color="gray")

# Etiqueta de espaciamiento vertical de receptores (25 m)
if len(receiver_z_m) > 1:
    ax.plot([10.0, 10.0], [receiver_z_m[0], receiver_z_m[1]], color="gray", lw=1.0)
    ax.text(14.0, (receiver_z_m[0] + receiver_z_m[1]) / 2.0, f"{receiver_spacing_m:.0f} m", va="center", color="gray")

# Formato: misma escala vertical que los logs
ax.set_ylim(0.0, 3800.0)
ax.invert_yaxis()
ax.set_xlim(-40.0, max(120.0, source_x_m + 40.0))
ax.set_xlabel("X (m)")
ax.set_ylabel("Profundidad Z (m)")
ax.set_title("Croquis geometría VSP")

for y in np.arange(0.0, 3800.0 + 500.0, 500.0):
    ax.axhline(y, color="gray", lw=0.6, alpha=0.4)

ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Celda 14: tiempos de llegada de onda directa (ray tracing sencillo)
# t_direct = distancia_fuente_receptor / velocidad_media(0->z_receptor)

# Perfil de velocidad y geometría
vel_profile_df = rv_model_df[["DEPT", "VEL"]].sort_values("DEPT").reset_index(drop=True)
z_nodes = vel_profile_df["DEPT"].to_numpy(dtype=float)
v_nodes = vel_profile_df["VEL"].to_numpy(dtype=float)

receiver_depths = vsp_geometry["receiver_z_m"].astype(float)
source_offset = float(vsp_geometry["source_offset_m"])

# Interpola velocidad en profundidades de receptores
v_rec = np.interp(receiver_depths, z_nodes, v_nodes)

# Tiempo vertical acumulado t(z)=integral dz/v usando regla trapezoidal
z_ext = np.unique(np.concatenate([z_nodes, receiver_depths]))
z_ext.sort()
v_ext = np.interp(z_ext, z_nodes, v_nodes)

# dt por intervalo y tiempo acumulado
z_diff = np.diff(z_ext)
v_mid = 0.5 * (v_ext[1:] + v_ext[:-1])
dt_seg = z_diff / np.maximum(v_mid, 1.0)
t_cum = np.concatenate([[0.0], np.cumsum(dt_seg)])

# Interpola tiempo vertical en receptores y calcula velocidad media
# v_avg(0->z) = z / t_vertical(z)
t_vertical_rec = np.interp(receiver_depths, z_ext, t_cum)
v_avg_rec = np.where(t_vertical_rec > 0.0, receiver_depths / t_vertical_rec, v_rec)

# Distancia fuente-receptor (pitágoras) y tiempo directo
path_length_m = np.sqrt(receiver_depths**2 + source_offset**2)
t_direct_s = path_length_m / np.maximum(v_avg_rec, 1.0)

direct_arrivals_df = pd.DataFrame(
    {
        "receiver_id": np.arange(len(receiver_depths), dtype=int),
        "z_receiver_m": receiver_depths,
        "x_source_m": source_offset,
        "path_length_m": path_length_m,
        "t_vertical_s": t_vertical_rec,
        "v_avg_mps": v_avg_rec,
        "t_direct_s": t_direct_s,
    }
)

print("Resumen onda directa:")
display(direct_arrivals_df[["z_receiver_m", "t_direct_s", "v_avg_mps"]].head(10))
print(f"N receptores: {len(direct_arrivals_df)}")
print(f"Rango tiempos directos: {direct_arrivals_df['t_direct_s'].min():.4f} s - {direct_arrivals_df['t_direct_s'].max():.4f} s")

In [ ]:
# Celda 15: plot profundidad-tiempo de onda directa
import matplotlib.pyplot as plt

df_direct = direct_arrivals_df.sort_values("z_receiver_m").copy()

depth = df_direct["z_receiver_m"].to_numpy()
time_s = df_direct["t_direct_s"].to_numpy()

fig, ax = plt.subplots(figsize=(6, 10))
ax.plot(time_s, depth, color="#0b1f5b", lw=1.2, marker="o", ms=2.5)

ax.set_xlabel("Tiempo de llegada onda directa (s)")
ax.set_ylabel("Profundidad (m)")
ax.set_title("Par profundidad-tiempo (onda directa)")

# Escala vertical habitual
ax.set_ylim(0.0, 3800.0)
ax.invert_yaxis()
for y in np.arange(0.0, 3800.0 + 500.0, 500.0):
    ax.axhline(y, color="gray", lw=0.6, alpha=0.5)

ax.grid(True, axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 16: rellenar mallado con spikes de onda directa y convolver con wavelet
# Usa direct_arrivals_df (tiempos), wavefield_mesh/time_axis_s (mallado) y source_wavelet (celda 11)

# Copias de trabajo
spike_mesh = np.zeros_like(wavefield_mesh, dtype=float)

dt_s = float(time_axis_s[1] - time_axis_s[0])
t0_s = float(time_axis_s[0])

# Inserta spike=1 en la muestra temporal más cercana para cada receptor
for row in direct_arrivals_df.itertuples(index=False):
    rid = int(row.receiver_id)
    t_arrival_s = float(row.t_direct_s)
    it = int(np.round((t_arrival_s - t0_s) / dt_s))
    if 0 <= rid < spike_mesh.shape[0] and 0 <= it < spike_mesh.shape[1]:
        spike_mesh[rid, it] += 1.0

# Convolución horizontal (eje tiempo) con wavelet de la fuente
wavelet = vsp_geometry["source_wavelet"]
convolved_mesh = np.zeros_like(spike_mesh, dtype=float)
for irec in range(spike_mesh.shape[0]):
    convolved_mesh[irec, :] = np.convolve(spike_mesh[irec, :], wavelet, mode="same")

print("Wavefield directo generado:")
print(f"- Shape spike_mesh: {spike_mesh.shape}")
print(f"- Shape convolved_mesh: {convolved_mesh.shape}")
print(f"- Spikes no-cero: {np.count_nonzero(spike_mesh)}")

In [ ]:
# Celda 17: visualización del mallado de spikes y del mallado convolucionado
import matplotlib.pyplot as plt

zmin = float(receiver_depths_m.min())
zmax = float(receiver_depths_m.max())
extent = [0.0, 4300.0, zmax, zmin]  # [xmin, xmax, ymax, ymin] para eje Y invertido

guide_start = np.floor(zmin / 500.0) * 500.0
guide_stop = np.ceil(zmax / 500.0) * 500.0

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

# Panel 1: spikes
im0 = axes[0].imshow(
    spike_mesh,
    aspect="auto",
    cmap="gray_r",
    extent=extent,
    interpolation="nearest",
)
axes[0].set_title("Mallado onda directa (spikes)")
axes[0].set_xlabel("Tiempo (ms)")
axes[0].set_ylabel("Profundidad (m)")
axes[0].set_xlim(0.0, 4300.0)
axes[0].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[0].axhline(y, color="gray", lw=0.5, alpha=0.4)

# Panel 2: convolución con wavelet
vmax = np.nanmax(np.abs(convolved_mesh))
im1 = axes[1].imshow(
    convolved_mesh,
    aspect="auto",
    cmap="seismic",
    extent=extent,
    vmin=-vmax,
    vmax=vmax,
    interpolation="nearest",
)
axes[1].set_title("Mallado onda directa (convolucionado)")
axes[1].set_xlabel("Tiempo (ms)")
axes[1].set_xlim(0.0, 4300.0)
axes[1].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[1].axhline(y, color="gray", lw=0.5, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 18: resample a 3 m (AI filtrada, VEL_AVG, OWT) y cálculo de RC
# RC[i] = (AI[i+1]-AI[i]) / (AI[i+1]+AI[i])

# Base: profundidad + atributos del dataframe principal
base_df = rv_model_df[["DEPT", "AI_FILT", "VEL_AVG", "OWT_S"]].sort_values("DEPT").dropna().reset_index(drop=True)

# Mallado vertical a 3 m
dz_rc_m = 3.0
z_min = 0.0
z_max = float(base_df["DEPT"].max())
z_rc = np.arange(z_min, z_max + dz_rc_m, dz_rc_m, dtype=float)

# Resampleo por interpolación lineal al nuevo muestreo
ai_rc = np.interp(z_rc, base_df["DEPT"].to_numpy(), base_df["AI_FILT"].to_numpy())
vel_avg_rc = np.interp(z_rc, base_df["DEPT"].to_numpy(), base_df["VEL_AVG"].to_numpy())
owt_rc_s = np.interp(z_rc, base_df["DEPT"].to_numpy(), base_df["OWT_S"].to_numpy())

# Cálculo de RC sobre AI resampleada
ai_next = np.roll(ai_rc, -1)
denom = ai_next + ai_rc
rc = np.where(np.abs(denom) > 1e-12, (ai_next - ai_rc) / denom, 0.0)
rc[-1] = 0.0

rc_df = pd.DataFrame(
    {
        "DEPT": z_rc,
        "AI_FILT_RESAMP_3M": ai_rc,
        "VEL_AVG_RESAMP_3M": vel_avg_rc,
        "OWT_S_RESAMP_3M": owt_rc_s,
        "OWT_MS_RESAMP_3M": 1000.0 * owt_rc_s,
        "RC": rc,
    }
)

print("RC dataframe creado:")
print(f"- Muestreo vertical: {dz_rc_m:.1f} m")
print(f"- Resampleo: interpolación lineal")
print(f"- Nº muestras: {len(rc_df)}")
display(rc_df.head())

In [ ]:
# Celda 19: plots de AI/RC y de OWT/VEL_AVG resampleados (2 filas)
import matplotlib.pyplot as plt

# Fila 1: AI/RC
# Track 1: AI filtrada original
df_ai_orig = rv_model_df[["DEPT", "AI_FILT"]].sort_values("DEPT")
# Tracks resampleados
df_rc_plot = rc_df.sort_values("DEPT")

fig, axes = plt.subplots(2, 3, figsize=(14, 12), sharey=True)

# --- Fila 1 ---
# 1) AI filtrada original
axes[0, 0].plot(df_ai_orig["AI_FILT"].to_numpy(), df_ai_orig["DEPT"].to_numpy(), color="black", lw=1.0)
axes[0, 0].set_xlabel("AI filtrada (original)")
axes[0, 0].set_title("AI_FILT (dz original)")

# 2) AI filtrada resampleada 3 m
axes[0, 1].plot(df_rc_plot["AI_FILT_RESAMP_3M"].to_numpy(), df_rc_plot["DEPT"].to_numpy(), color="#0b1f5b", lw=1.0)
axes[0, 1].set_xlabel("AI filtrada (resample 3m)")
axes[0, 1].set_title("AI_FILT (dz=3 m)")

# 3) RC
axes[0, 2].plot(df_rc_plot["RC"].to_numpy(), df_rc_plot["DEPT"].to_numpy(), color="darkred", lw=1.0)
axes[0, 2].axvline(0.0, color="gray", lw=0.8, ls="--")
axes[0, 2].set_xlabel("RC")
axes[0, 2].set_title("Coef. Reflexión")

# --- Fila 2 ---
# 4) OWT resampleado
axes[1, 0].plot(df_rc_plot["OWT_MS_RESAMP_3M"].to_numpy(), df_rc_plot["DEPT"].to_numpy(), color="purple", lw=1.0)
axes[1, 0].set_xlabel("OWT (ms) resample 3m")
axes[1, 0].set_title("Tiempo vertical (OWT)")

# 5) VEL_AVG resampleada
axes[1, 1].plot(df_rc_plot["VEL_AVG_RESAMP_3M"].to_numpy(), df_rc_plot["DEPT"].to_numpy(), color="teal", lw=1.0)
axes[1, 1].set_xlabel("VEL_AVG (m/s) resample 3m")
axes[1, 1].set_title("Velocidad media")

# 6) Track vacío para mantener layout de 2 filas
axes[1, 2].axis("off")

# Escala vertical habitual
for row in axes:
    for ax in row:
        if ax.has_data():
            ax.set_ylim(0.0, 3800.0)
            ax.invert_yaxis()
            for y in np.arange(0.0, 3800.0 + 500.0, 500.0):
                ax.axhline(y, color="gray", lw=0.6, alpha=0.5)
            ax.grid(False)

axes[0, 0].set_ylabel("Profundidad (m)")
axes[1, 0].set_ylabel("Profundidad (m)")
plt.suptitle("Atributos resampleados a 3 m para reflexiones primarias", y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Celda 20: tiempos de reflexiones primarias por receptor (2 tramos oblicuos)
# Criterio pedido:
# - Se usa VEL_AVG del reflector (hasta superficie) para ambos tramos.
# - Se considera oblicuidad por geometría (fuente a 60 m del pozo).
# - Rayos rectos y reflexión especular simplificada en reflector horizontal.

# Base de reflectores (resample 3 m)
reflectors_df = rc_df[["DEPT", "RC", "VEL_AVG_RESAMP_3M"]].copy()
reflectors_df = reflectors_df.sort_values("DEPT").reset_index(drop=True)

# Geometría
x_s = float(vsp_geometry["source_x_m"])      # fuente
x_r = float(vsp_geometry["wellhead_x_m"])    # pozo/receptores (x=0)
receiver_depths = np.asarray(vsp_geometry["receiver_z_m"], dtype=float)

# Para reflector horizontal y rayo especular: punto de reflexión en el punto medio horizontal
x_ref = 0.5 * (x_s + x_r)

primary_reflections_by_receiver = {}

for irec, z_rec in enumerate(receiver_depths):
    rows = []

    for row in reflectors_df.itertuples(index=False):
        z_ref = float(row.DEPT)
        rc = float(row.RC)
        v_ref_avg = float(row.VEL_AVG_RESAMP_3M)  # velocidad media 0->z_ref

        # Solo reflectores por debajo del receptor para primarias VSP
        if z_ref <= z_rec:
            continue

        # Tramo 1 (fuente -> reflector), oblicuo
        dz1 = z_ref - 0.0
        dx1 = abs(x_s - x_ref)
        l1_m = np.sqrt(dz1**2 + dx1**2)
        t1_s = l1_m / max(v_ref_avg, 1.0)

        # Tramo 2 (reflector -> receptor), oblicuo
        dz2 = z_ref - z_rec
        dx2 = abs(x_ref - x_r)
        l2_m = np.sqrt(dz2**2 + dx2**2)
        t2_s = l2_m / max(v_ref_avg, 1.0)

        t_primary_s = t1_s + t2_s

        rows.append(
            {
                "receiver_id": int(irec),
                "z_receiver_m": float(z_rec),
                "z_reflector_m": z_ref,
                "x_reflector_m": float(x_ref),
                "v_ref_avg_mps": float(v_ref_avg),
                "path1_m": float(l1_m),
                "path2_m": float(l2_m),
                "t1_s": float(t1_s),
                "t2_s": float(t2_s),
                "t_primary_s": float(t_primary_s),
                "RC": rc,
            }
        )

    primary_reflections_by_receiver[int(irec)] = pd.DataFrame(rows)

# Tabla consolidada (opcional)
primary_reflections_df = pd.concat(primary_reflections_by_receiver.values(), ignore_index=True)

print("Reflexiones primarias calculadas:")
print(f"- Receptores procesados: {len(primary_reflections_by_receiver)}")
print(f"- Eventos primarios totales: {len(primary_reflections_df)}")
print(f"- x_ref especular: {x_ref:.2f} m")

display(primary_reflections_df.head())

In [ ]:
# Celda 21: mallado de reflexiones primarias (spikes RC) + convolución
# Regla de llenado:
# - Si varios eventos caen en la misma celda (receptor, tiempo), usar promedio de RC.
# - Si no cae ninguno, valor 0.

# Mallados de trabajo
primary_spike_mesh = np.zeros_like(wavefield_mesh, dtype=float)
accum_mesh = np.zeros_like(wavefield_mesh, dtype=float)
count_mesh = np.zeros_like(wavefield_mesh, dtype=float)

dt_s = float(time_axis_s[1] - time_axis_s[0])
t0_s = float(time_axis_s[0])

# Acumula RC por celda (receiver_id, time_idx)
for row in primary_reflections_df.itertuples(index=False):
    rid = int(row.receiver_id)
    t_evt = float(row.t_primary_s)
    amp = float(row.RC)

    it = int(np.round((t_evt - t0_s) / dt_s))
    if 0 <= rid < accum_mesh.shape[0] and 0 <= it < accum_mesh.shape[1]:
        accum_mesh[rid, it] += amp
        count_mesh[rid, it] += 1.0

# Promedio en celdas con múltiples contribuciones
mask = count_mesh > 0.0
primary_spike_mesh[mask] = accum_mesh[mask] / count_mesh[mask]
primary_spike_mesh[~mask] = 0.0

# Convolución horizontal (eje tiempo) con wavelet de fuente
wavelet = vsp_geometry["source_wavelet"]
primary_convolved_mesh = np.zeros_like(primary_spike_mesh, dtype=float)
for irec in range(primary_spike_mesh.shape[0]):
    primary_convolved_mesh[irec, :] = np.convolve(primary_spike_mesh[irec, :], wavelet, mode="same")

print("Mallado de primarias generado:")
print(f"- Shape spikes: {primary_spike_mesh.shape}")
print(f"- Shape convolved: {primary_convolved_mesh.shape}")
print(f"- Celdas con eventos: {int(mask.sum())}")
print(f"- Máximo nº de eventos apilados en una celda: {int(count_mesh.max())}")

In [ ]:
# Celda 22: visualización de primarias (spikes RC y convolucionado)
import matplotlib.pyplot as plt

zmin = float(receiver_depths_m.min())
zmax = float(receiver_depths_m.max())
extent = [0.0, 4300.0, zmax, zmin]  # [xmin, xmax, ymax, ymin]

guide_start = np.floor(zmin / 500.0) * 500.0
guide_stop = np.ceil(zmax / 500.0) * 500.0

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

# Panel 1: spikes de RC
vmax_spk = np.nanmax(np.abs(primary_spike_mesh))
if vmax_spk <= 0:
    vmax_spk = 1.0

axes[0].imshow(
    primary_spike_mesh,
    aspect="auto",
    cmap="seismic",
    extent=extent,
    vmin=-vmax_spk,
    vmax=vmax_spk,
    interpolation="nearest",
)
axes[0].set_title("Primarias (spikes RC)")
axes[0].set_xlabel("Tiempo (ms)")
axes[0].set_ylabel("Profundidad (m)")
axes[0].set_xlim(0.0, 4300.0)
axes[0].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[0].axhline(y, color="gray", lw=0.5, alpha=0.4)

# Panel 2: primarias convolucionadas
vmax_conv = np.nanmax(np.abs(primary_convolved_mesh))
if vmax_conv <= 0:
    vmax_conv = 1.0

axes[1].imshow(
    primary_convolved_mesh,
    aspect="auto",
    cmap="seismic",
    extent=extent,
    vmin=-vmax_conv,
    vmax=vmax_conv,
    interpolation="nearest",
)
axes[1].set_title("Primarias (convolucionado)")
axes[1].set_xlabel("Tiempo (ms)")
axes[1].set_xlim(0.0, 4300.0)
axes[1].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[1].axhline(y, color="gray", lw=0.5, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 23: AI filtrado + gather total convolucionado (directas + primarias)
import matplotlib.pyplot as plt

# Track AI filtrado
df_ai = rv_model_df[["DEPT", "AI_FILT"]].sort_values("DEPT")

# Gather total convolucionado
# (mismo mallado receptor-tiempo: suma de directas y primarias)
total_convolved_mesh = convolved_mesh + primary_convolved_mesh

zmin = float(receiver_depths_m.min())
zmax = float(receiver_depths_m.max())
guide_start = np.floor(zmin / 500.0) * 500.0
guide_stop = np.ceil(zmax / 500.0) * 500.0

fig, axes = plt.subplots(1, 2, figsize=(13, 9), gridspec_kw={"width_ratios": [1, 2]}, sharey=True)

# Izquierda: AI filtrado
axes[0].plot(df_ai["AI_FILT"].to_numpy(), df_ai["DEPT"].to_numpy(), color="black", lw=1.0)
axes[0].set_title("AI filtrado")
axes[0].set_xlabel("AI_FILT")
axes[0].set_ylabel("Profundidad (m)")

# Derecha: gather total convolucionado
vmax_tot = np.nanmax(np.abs(total_convolved_mesh))
if vmax_tot <= 0:
    vmax_tot = 1.0

extent = [0.0, 4300.0, zmax, zmin]
axes[1].imshow(
    total_convolved_mesh,
    aspect="auto",
    cmap="seismic",
    extent=extent,
    vmin=-vmax_tot,
    vmax=vmax_tot,
    interpolation="nearest",
)
axes[1].set_title("Gather convolucionado: directas + primarias")
axes[1].set_xlabel("Tiempo (ms)")
axes[1].set_xlim(0.0, 4300.0)
axes[1].set_ylim(zmax, zmin)

# Escala vertical real de receptores en ambos paneles
for ax in axes:
    ax.set_ylim(zmax, zmin)
    for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
        ax.axhline(y, color="gray", lw=0.5, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 24: múltiplos downgoing (3 tramos) en diccionario por receptor
# Esquema:
#   1) fuente -> reflector profundo
#   2) reflector profundo -> reflector somero
#   3) reflector somero -> receptor
# Hipótesis simplificadas:
# - Rayos rectos.
# - Reflexión especular en cada interfaz (Snell local en interfaz horizontal).
# - Interfaces tomadas de rc_df (resample 3 m).

# Base de interfaces (reflectores)
interfaces_df = rc_df[["DEPT", "RC", "VEL_AVG_RESAMP_3M"]].sort_values("DEPT").reset_index(drop=True)
interfaces_df.at[0, "RC"] = 1

# Geometría
x_s = float(vsp_geometry["source_x_m"])      # fuente
x_r = float(vsp_geometry["wellhead_x_m"])    # pozo/receptores
receiver_depths = np.asarray(vsp_geometry["receiver_z_m"], dtype=float)

# Reflexión especular encadenada (2 rebotes):
# x_deep = (2*x_s + x_r)/3, x_shallow = (x_s + 2*x_r)/3
x_deep = (2.0 * x_s + x_r) / 3.0
x_shallow = (x_s + 2.0 * x_r) / 3.0

downgoing_multiples_by_receiver = {}

for irec, z_rec in enumerate(receiver_depths):
    rows = []

    # Reflector profundo: por debajo del receptor
    deep_candidates = interfaces_df[interfaces_df["DEPT"] > z_rec]
    # Reflector somero: por encima del receptor
    shallow_candidates = interfaces_df[interfaces_df["DEPT"] < z_rec]

    for deep_row in deep_candidates.itertuples(index=False):
        z_deep = float(deep_row.DEPT)
        rc_deep = float(deep_row.RC)
        v_deep = float(deep_row.VEL_AVG_RESAMP_3M)

        for sh_row in shallow_candidates.itertuples(index=False):
            z_sh = float(sh_row.DEPT)
            rc_sh = float(sh_row.RC)
            v_sh = float(sh_row.VEL_AVG_RESAMP_3M)

            # Geometría física del downgoing múltiple: z_deep > z_sh
            if z_deep <= z_sh:
                continue

            # Tramo 1: fuente -> reflector profundo
            l1_m = np.sqrt((z_deep - 0.0) ** 2 + (x_s - x_deep) ** 2)
            t1_s = l1_m / max(v_deep, 1.0)

            # Tramo 2: reflector profundo -> reflector somero
            # Velocidad efectiva simple: promedio de v_avg profundo y somero
            v_seg2 = 0.5 * (v_deep + v_sh)
            l2_m = np.sqrt((z_deep - z_sh) ** 2 + (x_deep - x_shallow) ** 2)
            t2_s = l2_m / max(v_seg2, 1.0)

            # Tramo 3: reflector somero -> receptor
            l3_m = np.sqrt((z_rec - z_sh) ** 2 + (x_r - x_shallow) ** 2)
            t3_s = l3_m / max(v_sh, 1.0)

            t_downgoing_s = t1_s + t2_s + t3_s

            # Amplitud solicitada por el usuario (forma literal simplificada)
            # rc_downgoing = (1.0 + rc_sh) * rc_deep * (1.0 - rc_sh * rc_sh) * (1.0 + rc_sh)
            rc_downgoing = rc_deep * (- rc_sh)

            rows.append(
                {
                    "receiver_id": int(irec),
                    "z_receiver_m": float(z_rec),
                    "z_reflector_deep_m": z_deep,
                    "z_reflector_shallow_m": z_sh,
                    "x_reflector_deep_m": float(x_deep),
                    "x_reflector_shallow_m": float(x_shallow),
                    "path1_m": float(l1_m),
                    "path2_m": float(l2_m),
                    "path3_m": float(l3_m),
                    "t1_s": float(t1_s),
                    "t2_s": float(t2_s),
                    "t3_s": float(t3_s),
                    "t_downgoing_s": float(t_downgoing_s),
                    "RC_deep": rc_deep,
                    "RC_shallow": rc_sh,
                    "RC_downgoing": float(rc_downgoing),
                }
            )

    downgoing_multiples_by_receiver[int(irec)] = pd.DataFrame(rows)

# Tabla consolidada opcional
downgoing_multiples_df = pd.concat(downgoing_multiples_by_receiver.values(), ignore_index=True)

print("Múltiplos downgoing calculados:")
print(f"- Receptores procesados: {len(downgoing_multiples_by_receiver)}")
print(f"- Eventos totales: {len(downgoing_multiples_df)}")
print(f"- x_ref profundo (especular): {x_deep:.2f} m")
print(f"- x_ref somero (especular): {x_shallow:.2f} m")

display(downgoing_multiples_df.head())

In [ ]:
# Celda 25: mallado de múltiplos downgoing (spikes RC) + convolución
# Regla de llenado:
# - Si varios eventos caen en la misma celda (receptor, tiempo), usar promedio de RC_downgoing.
# - Si no cae ninguno, valor 0.

# Mallados de trabajo
down_spike_mesh = np.zeros_like(wavefield_mesh, dtype=float)
accum_down_mesh = np.zeros_like(wavefield_mesh, dtype=float)
count_down_mesh = np.zeros_like(wavefield_mesh, dtype=float)

dt_s = float(time_axis_s[1] - time_axis_s[0])
t0_s = float(time_axis_s[0])

# Acumula RC downgoing por celda (receiver_id, time_idx)
for row in downgoing_multiples_df.itertuples(index=False):
    rid = int(row.receiver_id)
    t_evt = float(row.t_downgoing_s)
    amp = float(row.RC_downgoing)

    it = int(np.round((t_evt - t0_s) / dt_s))
    if 0 <= rid < accum_down_mesh.shape[0] and 0 <= it < accum_down_mesh.shape[1]:
        accum_down_mesh[rid, it] += amp
        count_down_mesh[rid, it] += 1.0

# Promedio en celdas con múltiples contribuciones
mask_down = count_down_mesh > 0.0
down_spike_mesh[mask_down] = accum_down_mesh[mask_down] / count_down_mesh[mask_down]
down_spike_mesh[~mask_down] = 0.0

# Convolución horizontal (eje tiempo) con wavelet de fuente
wavelet = vsp_geometry["source_wavelet"]
down_convolved_mesh = np.zeros_like(down_spike_mesh, dtype=float)
for irec in range(down_spike_mesh.shape[0]):
    down_convolved_mesh[irec, :] = np.convolve(down_spike_mesh[irec, :], wavelet, mode="same")

print("Mallado downgoing generado:")
print(f"- Shape spikes: {down_spike_mesh.shape}")
print(f"- Shape convolved: {down_convolved_mesh.shape}")
print(f"- Celdas con eventos: {int(mask_down.sum())}")
print(f"- Máximo nº de eventos apilados en una celda: {int(count_down_mesh.max())}")

In [ ]:
# (Celda eliminada por petición del usuario)
# Objetivo: anular amplitudes espurias al final del registro por receptor.
# Por defecto se anulan las últimas 3 muestras (3 ms).


In [ ]:
# Celda 26: visualización de downgoing (spikes RC y convolucionado)
import matplotlib.pyplot as plt

zmin = float(receiver_depths_m.min())
zmax = float(receiver_depths_m.max())
extent = [0.0, 4300.0, zmax, zmin]  # [xmin, xmax, ymax, ymin]

guide_start = np.floor(zmin / 500.0) * 500.0
guide_stop = np.ceil(zmax / 500.0) * 500.0

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

# Panel 1: spikes de RC downgoing
vmax_spk = np.nanpercentile(np.abs(down_spike_mesh), 99.0)
if vmax_spk <= 0:
    vmax_spk = 1.0

axes[0].imshow(
    down_spike_mesh,
    aspect="auto",
    cmap="RdBu_r",
    extent=extent,
    vmin=-vmax_spk,
    vmax=vmax_spk,
    interpolation="nearest",
)
axes[0].set_title("Downgoing (spikes RC)")
axes[0].set_xlabel("Tiempo (ms)")
axes[0].set_ylabel("Profundidad (m)")
axes[0].set_xlim(0.0, 4300.0)
axes[0].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[0].axhline(y, color="gray", lw=0.5, alpha=0.4)

# Panel 2: downgoing convolucionadas
vmax_conv = np.nanpercentile(np.abs(down_convolved_mesh), 99.0)
if vmax_conv <= 0:
    vmax_conv = 1.0

axes[1].imshow(
    down_convolved_mesh,
    aspect="auto",
    cmap="RdBu_r",
    extent=extent,
    vmin=-vmax_conv,
    vmax=vmax_conv,
    interpolation="nearest",
)
axes[1].set_title("Downgoing (convolucionado)")
axes[1].set_xlabel("Tiempo (ms)")
axes[1].set_xlim(0.0, 4300.0)
axes[1].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[1].axhline(y, color="gray", lw=0.5, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 27: AI filtrado + gather total convolucionado (directas + primarias + downgoing)
import matplotlib.pyplot as plt

# Track AI filtrado
df_ai = rv_model_df[["DEPT", "AI_FILT"]].sort_values("DEPT")

# Gather total convolucionado
total_convolved_mesh = convolved_mesh + primary_convolved_mesh + 100*down_convolved_mesh

zmin = float(receiver_depths_m.min())
zmax = float(receiver_depths_m.max())
extent = [0.0, 4300.0, zmax, zmin]

guide_start = np.floor(zmin / 500.0) * 500.0
guide_stop = np.ceil(zmax / 500.0) * 500.0

fig, axes = plt.subplots(1, 2, figsize=(13, 9), gridspec_kw={"width_ratios": [1, 2]}, sharey=True)

# Izquierda: AI filtrado
axes[0].plot(df_ai["AI_FILT"].to_numpy(), df_ai["DEPT"].to_numpy(), color="black", lw=1.0)
axes[0].set_title("AI filtrado")
axes[0].set_xlabel("AI_FILT")
axes[0].set_ylabel("Profundidad (m)")

# Derecha: gather total convolucionado
vmax_conv = np.nanpercentile(np.abs(total_convolved_mesh), 99.0)
if vmax_conv <= 0:
    vmax_conv = 1.0

axes[1].imshow(
    total_convolved_mesh,
    aspect="auto",
    cmap="RdBu_r",
    extent=extent,
    vmin=-vmax_conv,
    vmax=vmax_conv,
    interpolation="nearest",
)
axes[1].set_title("Total convolucionado: directas + primarias + downgoing")
axes[1].set_xlabel("Tiempo (ms)")
axes[1].set_xlim(0.0, 4300.0)
axes[1].set_ylim(zmax, zmin)

# Escala vertical real en ambos paneles
for ax in axes:
    ax.set_ylim(zmax, zmin)
    for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
        ax.axhline(y, color="gray", lw=0.5, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 28: múltiplos upgoing (4 tramos) en diccionario por receptor
# Esquema solicitado:
#   1) Fuente -> ReflectorDeep
#   2) ReflectorDeep -> ReflectorShallow
#   3) ReflectorShallow -> ReflectorDeep
#   4) ReflectorDeep -> Receptor
# Hipótesis simplificadas:
# - Ray tracing recto por tramos.
# - Reflexión especular local en interfaces horizontales.
# - Interfaces tomadas de rc_df (resample 3 m).
# - Velocidad de todos los tramos: VEL_AVG del reflector deep (0 -> z_deep).

interfaces_df = rc_df[["DEPT", "RC", "VEL_AVG_RESAMP_3M"]].sort_values("DEPT").reset_index(drop=True)

# Geometría
x_s = float(vsp_geometry["source_x_m"])
x_r = float(vsp_geometry["wellhead_x_m"])
receiver_depths = np.asarray(vsp_geometry["receiver_z_m"], dtype=float)

# Puntos horizontales de rebote (aprox. especular simplificada para 4 tramos)
# partición uniforme del offset fuente-receptor
x_d1 = 0.75 * x_s + 0.25 * x_r
x_sh = 0.50 * x_s + 0.50 * x_r
x_d2 = 0.25 * x_s + 0.75 * x_r

upgoing_multiples_by_receiver = {}

for irec, z_rec in enumerate(receiver_depths):
    rows = []

    # Reflector deep: por debajo del receptor
    deep_candidates = interfaces_df[interfaces_df["DEPT"] > z_rec]
    # Reflector shallow: por encima del receptor
    shallow_candidates = interfaces_df[interfaces_df["DEPT"] < z_rec]

    for deep_row in deep_candidates.itertuples(index=False):
        z_deep = float(deep_row.DEPT)
        rc_deep = float(deep_row.RC)
        v_deep = float(deep_row.VEL_AVG_RESAMP_3M)

        for sh_row in shallow_candidates.itertuples(index=False):
            z_sh = float(sh_row.DEPT)
            rc_sh = float(sh_row.RC)

            # Consistencia geométrica
            if z_deep <= z_sh:
                continue

            # Tramo 1: Fuente -> Deep
            l1_m = np.sqrt((z_deep - 0.0) ** 2 + (x_s - x_d1) ** 2)
            # Tramo 2: Deep -> Shallow
            l2_m = np.sqrt((z_deep - z_sh) ** 2 + (x_d1 - x_sh) ** 2)
            # Tramo 3: Shallow -> Deep
            l3_m = np.sqrt((z_deep - z_sh) ** 2 + (x_sh - x_d2) ** 2)
            # Tramo 4: Deep -> Receptor
            l4_m = np.sqrt((z_deep - z_rec) ** 2 + (x_d2 - x_r) ** 2)

            # Velocidad pedida: v_avg hasta reflector deep para todos los tramos
            t1_s = l1_m / max(v_deep, 1.0)
            t2_s = l2_m / max(v_deep, 1.0)
            t3_s = l3_m / max(v_deep, 1.0)
            t4_s = l4_m / max(v_deep, 1.0)
            t_upgoing_s = t1_s + t2_s + t3_s + t4_s

            # RC pedido: RCdeep * (-RCshallow) * RCdeep
            rc_upgoing = rc_deep * (-rc_sh) * rc_deep

            rows.append(
                {
                    "receiver_id": int(irec),
                    "z_receiver_m": float(z_rec),
                    "z_reflector_deep_m": z_deep,
                    "z_reflector_shallow_m": z_sh,
                    "x_reflector_deep_1_m": float(x_d1),
                    "x_reflector_shallow_m": float(x_sh),
                    "x_reflector_deep_2_m": float(x_d2),
                    "v_deep_avg_mps": float(v_deep),
                    "path1_m": float(l1_m),
                    "path2_m": float(l2_m),
                    "path3_m": float(l3_m),
                    "path4_m": float(l4_m),
                    "t1_s": float(t1_s),
                    "t2_s": float(t2_s),
                    "t3_s": float(t3_s),
                    "t4_s": float(t4_s),
                    "t_upgoing_s": float(t_upgoing_s),
                    "RC_deep": rc_deep,
                    "RC_shallow": rc_sh,
                    "RC_upgoing": float(rc_upgoing),
                }
            )

    upgoing_multiples_by_receiver[int(irec)] = pd.DataFrame(rows)

upgoing_multiples_df = pd.concat(upgoing_multiples_by_receiver.values(), ignore_index=True)

print("Múltiplos upgoing calculados:")
print(f"- Receptores procesados: {len(upgoing_multiples_by_receiver)}")
print(f"- Eventos totales: {len(upgoing_multiples_df)}")
print(f"- x_d1={x_d1:.2f} m, x_sh={x_sh:.2f} m, x_d2={x_d2:.2f} m")

display(upgoing_multiples_df.head())

In [ ]:
# Celda 29: mallado de múltiplos upgoing (spikes RC) + convolución
# Regla de llenado:
# - Si varios eventos caen en la misma celda (receptor, tiempo), usar promedio de RC_upgoing.
# - Si no cae ninguno, valor 0.

up_spike_mesh = np.zeros_like(wavefield_mesh, dtype=float)
accum_up_mesh = np.zeros_like(wavefield_mesh, dtype=float)
count_up_mesh = np.zeros_like(wavefield_mesh, dtype=float)

dt_s = float(time_axis_s[1] - time_axis_s[0])
t0_s = float(time_axis_s[0])

# Acumula RC upgoing por celda (receiver_id, time_idx)
for row in upgoing_multiples_df.itertuples(index=False):
    rid = int(row.receiver_id)
    t_evt = float(row.t_upgoing_s)
    amp = float(row.RC_upgoing)

    it = int(np.round((t_evt - t0_s) / dt_s))
    if 0 <= rid < accum_up_mesh.shape[0] and 0 <= it < accum_up_mesh.shape[1]:
        accum_up_mesh[rid, it] += amp
        count_up_mesh[rid, it] += 1.0

# Promedio en celdas con múltiples contribuciones
mask_up = count_up_mesh > 0.0
up_spike_mesh[mask_up] = accum_up_mesh[mask_up] / count_up_mesh[mask_up]
up_spike_mesh[~mask_up] = 0.0

# Convolución horizontal (eje tiempo) con wavelet de fuente
wavelet = vsp_geometry["source_wavelet"]
up_convolved_mesh = np.zeros_like(up_spike_mesh, dtype=float)
for irec in range(up_spike_mesh.shape[0]):
    up_convolved_mesh[irec, :] = np.convolve(up_spike_mesh[irec, :], wavelet, mode="same")

print("Mallado upgoing generado:")
print(f"- Shape spikes: {up_spike_mesh.shape}")
print(f"- Shape convolved: {up_convolved_mesh.shape}")
print(f"- Celdas con eventos: {int(mask_up.sum())}")
print(f"- Máximo nº de eventos apilados en una celda: {int(count_up_mesh.max())}")

In [ ]:
# Celda 30: visualización de upgoing (spikes RC y convolucionado)
import matplotlib.pyplot as plt

zmin = float(receiver_depths_m.min())
zmax = float(receiver_depths_m.max())
extent = [0.0, 4300.0, zmax, zmin]  # [xmin, xmax, ymax, ymin]

guide_start = np.floor(zmin / 500.0) * 500.0
guide_stop = np.ceil(zmax / 500.0) * 500.0

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

# Panel 1: spikes de RC upgoing
vmax_spk = np.nanpercentile(np.abs(up_spike_mesh), 99.0)
if vmax_spk <= 0:
    vmax_spk = 1.0

axes[0].imshow(
    up_spike_mesh,
    aspect="auto",
    cmap="RdBu_r",
    extent=extent,
    vmin=-vmax_spk,
    vmax=vmax_spk,
    interpolation="nearest",
)
axes[0].set_title("Upgoing (spikes RC)")
axes[0].set_xlabel("Tiempo (ms)")
axes[0].set_ylabel("Profundidad (m)")
axes[0].set_xlim(0.0, 4300.0)
axes[0].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[0].axhline(y, color="gray", lw=0.5, alpha=0.4)

# Panel 2: upgoing convolucionadas
vmax_conv = np.nanpercentile(np.abs(up_convolved_mesh), 99.0)
if vmax_conv <= 0:
    vmax_conv = 1.0

axes[1].imshow(
    up_convolved_mesh,
    aspect="auto",
    cmap="RdBu_r",
    extent=extent,
    vmin=-vmax_conv,
    vmax=vmax_conv,
    interpolation="nearest",
)
axes[1].set_title("Upgoing (convolucionado)")
axes[1].set_xlabel("Tiempo (ms)")
axes[1].set_xlim(0.0, 4300.0)
axes[1].set_ylim(zmax, zmin)
for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
    axes[1].axhline(y, color="gray", lw=0.5, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 31: AI filtrado + gather total convolucionado (directas + primarias + downgoing + upgoing)
import matplotlib.pyplot as plt

# Track AI filtrado
df_ai = rv_model_df[["DEPT", "AI_FILT"]].sort_values("DEPT")

# Suma total de componentes convolucionadas
total_all_convolved_mesh = (
    convolved_mesh
    + primary_convolved_mesh
    + 100*down_convolved_mesh
    + 500*up_convolved_mesh
)

zmin = float(receiver_depths_m.min())
zmax = float(receiver_depths_m.max())
extent = [0.0, 4300.0, zmax, zmin]

guide_start = np.floor(zmin / 500.0) * 500.0
guide_stop = np.ceil(zmax / 500.0) * 500.0

fig, axes = plt.subplots(1, 2, figsize=(13, 9), gridspec_kw={"width_ratios": [1, 2]}, sharey=True)

# Izquierda: AI filtrado
axes[0].plot(df_ai["AI_FILT"].to_numpy(), df_ai["DEPT"].to_numpy(), color="black", lw=1.0)
axes[0].set_title("AI filtrado")
axes[0].set_xlabel("AI_FILT")
axes[0].set_ylabel("Profundidad (m)")

# Derecha: gather total convolucionado
vmax_tot = np.nanpercentile(np.abs(total_all_convolved_mesh), 99.0)
if vmax_tot <= 0:
    vmax_tot = 1.0

axes[1].imshow(
    total_all_convolved_mesh,
    aspect="auto",
    cmap="RdBu_r",
    extent=extent,
    vmin=-vmax_tot,
    vmax=vmax_tot,
    interpolation="nearest",
)
axes[1].set_title("Total convolucionado: directas + primarias + downgoing + upgoing")
axes[1].set_xlabel("Tiempo (ms)")
axes[1].set_xlim(0.0, 4300.0)
axes[1].set_ylim(zmax, zmin)

# Escala vertical real en ambos paneles
for ax in axes:
    ax.set_ylim(zmax, zmin)
    for y in np.arange(guide_start, guide_stop + 500.0, 500.0):
        ax.axhline(y, color="gray", lw=0.5, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Celda 32: exportar gather total convolucionado a SEGY
from pathlib import Path

import subprocess
import sys
import numpy as np


def _pip_install(pkg_name):
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg_name],
        capture_output=True,
        text=True,
        check=False,
    )
    return proc.returncode, (proc.stdout or "")[-600:], (proc.stderr or "")[-600:]


# Preferencia: segyio; fallback: obspy
try:
    import segyio
    _segy_backend = "segyio"
except Exception:
    segyio = None
    _segy_backend = None

    code, out_tail, err_tail = _pip_install("segyio")

    try:
        import segyio
        _segy_backend = "segyio"
    except Exception:
        segyio = None
        _segy_backend = None

try:
    from obspy import Stream, Trace, read as obspy_read
    from obspy.io.segy.segy import SEGYBinaryFileHeader, SEGYTextualFileHeader, SEGYTraceHeader
    _obspy_available = True
except Exception:
    Stream = Trace = obspy_read = None
    SEGYBinaryFileHeader = SEGYTextualFileHeader = SEGYTraceHeader = None
    _obspy_available = False

    if _segy_backend is None:
        code, out_tail, err_tail = _pip_install("obspy")

        try:
            from obspy import Stream, Trace, read as obspy_read
            from obspy.io.segy.segy import SEGYBinaryFileHeader, SEGYTextualFileHeader, SEGYTraceHeader
            _obspy_available = True
        except Exception:
            Stream = Trace = obspy_read = None
            SEGYBinaryFileHeader = SEGYTextualFileHeader = SEGYTraceHeader = None
            _obspy_available = False

# -------------------- Validaciones de entrada --------------------
if "total_all_convolved_mesh" not in globals():
    raise NameError("No existe 'total_all_convolved_mesh'. Ejecuta primero la Celda 31.")
if "receiver_depths_m" not in globals():
    raise NameError("No existe 'receiver_depths_m'. Ejecuta primero la geometría VSP.")
if "time_axis_s" not in globals():
    raise NameError("No existe 'time_axis_s'. Ejecuta primero el mallado temporal.")

mesh = np.asarray(total_all_convolved_mesh, dtype=np.float32)
receiver_depths = np.asarray(receiver_depths_m, dtype=float)
time_s = np.asarray(time_axis_s, dtype=float)

if mesh.ndim != 2:
    raise ValueError(f"total_all_convolved_mesh debe ser 2D [n_receptores, n_tiempos], recibido: {mesh.shape}")

n_traces, n_samples = mesh.shape
if receiver_depths.shape[0] != n_traces:
    raise ValueError(
        f"No coincide n_receptores: mesh={n_traces} vs receiver_depths_m={receiver_depths.shape[0]}"
    )
if time_s.shape[0] != n_samples:
    raise ValueError(
        f"No coincide n_muestras: mesh={n_samples} vs time_axis_s={time_s.shape[0]}"
    )

if n_samples < 2:
    raise ValueError("Se requieren al menos 2 muestras de tiempo para definir dt.")

sample_interval_us = int(round((time_s[1] - time_s[0]) * 1_000_000.0))
if sample_interval_us != 1000:
    raise ValueError(f"El dt detectado no es 1 ms. Detectado: {sample_interval_us} us")

# Rutas de salida
output_path = Path("vsp_model_ray_tracing_total_convolved.sgy").resolve()
text_header = "VSP model with Ray Tracing"
receiver_spacing_m = 25.0

# -------------------- Escritura SEGY --------------------
if _segy_backend == "segyio":
    try:
        spec = segyio.spec()
        spec.sorting = segyio.TraceSortingFormat.UNKNOWN_SORTING
        spec.format = segyio.SegySampleFormat.IBM_FLOAT_4_BYTE
        spec.samples = np.arange(n_samples, dtype=np.int32)
        spec.tracecount = n_traces

        with segyio.create(str(output_path), spec) as f:
            text = (text_header + " " * 3200)[:3200]
            try:
                f.text[0] = segyio.tools.wrap(text)
            except Exception:
                f.text[0] = text.encode("ascii", errors="replace")

            f.bin[segyio.BinField.Interval] = sample_interval_us
            f.bin[segyio.BinField.Samples] = n_samples

            # Compatibilidad entre versiones de segyio para elevación de receptor
            depth_field_name = None
            depth_field = None
            for candidate in ["GroupElevation", "ReceiverGroupElevation"]:
                if hasattr(segyio.TraceField, candidate):
                    depth_field_name = candidate
                    depth_field = getattr(segyio.TraceField, candidate)
                    break

            if depth_field is None:
                raise AttributeError("segyio.TraceField no tiene GroupElevation ni ReceiverGroupElevation")

            for irec in range(n_traces):
                f.trace[irec] = mesh[irec, :].astype(np.float32)

                f.header[irec][segyio.TraceField.TRACE_SEQUENCE_LINE] = irec + 1
                f.header[irec][segyio.TraceField.TRACE_SEQUENCE_FILE] = irec + 1
                f.header[irec][segyio.TraceField.FieldRecord] = 1
                f.header[irec][segyio.TraceField.TraceNumber] = irec + 1
                f.header[irec][segyio.TraceField.TRACE_SAMPLE_INTERVAL] = sample_interval_us
                f.header[irec][segyio.TraceField.TRACE_SAMPLE_COUNT] = n_samples

                # Mapeo elegido: profundidad del receptor en GroupElevation (o alias compatible)
                f.header[irec][depth_field] = int(round(receiver_depths[irec]))

                # Geometría VSP básica (pozo vertical en x=0)
                f.header[irec][segyio.TraceField.SourceX] = 0
                f.header[irec][segyio.TraceField.GroupX] = 0
                f.header[irec][segyio.TraceField.SourceY] = 0
                f.header[irec][segyio.TraceField.GroupY] = 0

            f.flush()
    except Exception:
        raise

elif _obspy_available:
    st = Stream()
    for irec in range(n_traces):
        tr = Trace(data=mesh[irec, :].astype(np.float32))
        tr.stats.delta = 0.001
        tr.stats.starttime = 0

        tr.stats.segy = {}
        tr.stats.segy.trace_header = SEGYTraceHeader()
        tr.stats.segy.trace_header.trace_sequence_number_within_line = irec + 1
        tr.stats.segy.trace_header.trace_sequence_number_within_segy_file = irec + 1
        tr.stats.segy.trace_header.original_field_record_number = 1
        tr.stats.segy.trace_header.trace_number_within_the_original_field_record = irec + 1
        tr.stats.segy.trace_header.sample_interval_in_ms_for_this_trace = sample_interval_us
        tr.stats.segy.trace_header.number_of_samples_in_this_trace = n_samples
        tr.stats.segy.trace_header.group_elevation = int(round(receiver_depths[irec]))
        st.append(tr)

    st.stats = {}
    st.stats.textual_file_header = SEGYTextualFileHeader()
    text = (text_header + " " * 3200)[:3200]
    st.stats.textual_file_header.text = text.encode("ascii", errors="replace")

    st.stats.binary_file_header = SEGYBinaryFileHeader()
    st.stats.binary_file_header.sample_interval_in_microseconds = sample_interval_us
    st.stats.binary_file_header.number_of_samples_per_data_trace = n_samples

    st.write(str(output_path), format="SEGY", data_encoding=1)

else:
    raise ImportError("No se encontró backend SEGY. Instala 'segyio' u 'obspy'.")

# -------------------- Verificación post-escritura --------------------
verify_n_traces = None
verify_n_samples = None
verify_dt_us = None
verify_header_has_text = None

if _segy_backend == "segyio":
    with segyio.open(str(output_path), "r", ignore_geometry=True) as fchk:
        verify_n_traces = fchk.tracecount
        verify_n_samples = len(fchk.samples)
        verify_dt_us = int(fchk.bin[segyio.BinField.Interval])
        txt = bytes(fchk.text[0]).decode("ascii", errors="ignore")
        verify_header_has_text = text_header in txt
else:
    st_chk = obspy_read(str(output_path), format="SEGY")
    verify_n_traces = len(st_chk)
    verify_n_samples = int(st_chk[0].stats.npts) if len(st_chk) else 0
    verify_dt_us = int(round(st_chk[0].stats.delta * 1_000_000.0)) if len(st_chk) else None
    verify_header_has_text = True

print("SEGY exportado correctamente")
print(f"- Archivo: {output_path}")
print(f"- Shape mallado: {mesh.shape}")
print(f"- Trazas verificadas: {verify_n_traces}")
print(f"- Muestras verificadas por traza: {verify_n_samples}")
print(f"- dt verificado: {verify_dt_us} us")
print(f"- Header textual OK: {verify_header_has_text}")
print(f"- Profundidad receptores: {receiver_depths.min():.1f} m -> {receiver_depths.max():.1f} m")
print(f"- Espaciado receptores esperado: {receiver_spacing_m:.1f} m")